# Interval Data ETL and Service Level Analysis
## Blue Shield Call Center - 30-Minute Interval Analysis

This notebook:
1. ETLs complex pivot-table Excel files with LOB tabs
2. Transforms wide day-of-week format to long format
3. Analyzes service level performance vs goals
4. Visualizes deviation ranges by LOB

In [ ]:
# Parameters — update these paths for your engagement
data_dir = "~/plumb/books/sample-engagement/01-source/interval-data"
output_dir = "~/plumb/books/sample-engagement/02-processed/results"
sl_goal_default = 0.80
sl_goal_premium = 0.90  # For Concierge and Designated

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from datetime import datetime, timedelta

os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# Define LOB-specific service level goals
sl_goals = {
    'Concierge': sl_goal_premium,
    'Designated': sl_goal_premium,
    'Premier Core': sl_goal_default,
    'IFP PPO': sl_goal_default,
    'IFP TRIO': sl_goal_default,
    'Medicare': sl_goal_default,
    'MedSupp': sl_goal_default,
    'FEP Classic': sl_goal_default,
    'FEP Postal': sl_goal_default,
    'MediCal': sl_goal_default,
    'Small Group': sl_goal_default
}

print("Service Level Goals by LOB:")
for lob, goal in sl_goals.items():
    print(f"  {lob}: {goal*100:.0f}%")

In [ ]:
def parse_interval_sheet(df, lob_name, week_start):
    """
    Parse a single LOB sheet from the wide pivot format to long format.
    
    The sheet has:
    - Columns for each day (Sunday-Saturday)
    - Each day has 6 metrics: Actual SL, Forecast Calls, Actual Calls, Forecast AHT, Actual AHT, Actual Abandon
    - Rows are 30-minute intervals
    """
    days = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    metrics = ['Actual Service Level', 'Forecast Calls Offered', 'Actual Calls Offered', 
               'Forecast AHT', 'Actual AHT', 'Actual Abandon']
    
    records = []
    
    # Iterate through each row (interval)
    for idx, row in df.iterrows():
        interval = row.iloc[1]  # Interval(30) column
        
        if pd.isna(interval) or interval == 'Interval(30)':
            continue
            
        # Parse interval time
        try:
            if isinstance(interval, str):
                interval_time = datetime.strptime(interval, '%H:%M:%S').time()
            else:
                interval_time = interval
        except:
            continue
        
        # For each day of the week
        for day_idx, day in enumerate(days):
            col_start = 2 + (day_idx * 6)  # Starting column for this day
            
            # Calculate actual date
            actual_date = week_start + timedelta(days=day_idx)
            
            # Extract metrics for this day
            try:
                actual_sl = row.iloc[col_start]
                forecast_calls = row.iloc[col_start + 1]
                actual_calls = row.iloc[col_start + 2]
                forecast_aht = row.iloc[col_start + 3]
                actual_aht = row.iloc[col_start + 4]
                actual_abandon = row.iloc[col_start + 5]
                
                # Only add records with actual data
                if pd.notna(actual_sl) or pd.notna(actual_calls):
                    records.append({
                        'LOB': lob_name,
                        'Date': actual_date,
                        'Day': day,
                        'Interval': str(interval_time)[:5],
                        'Actual_SL': actual_sl if pd.notna(actual_sl) else None,
                        'Forecast_Calls': forecast_calls if pd.notna(forecast_calls) else 0,
                        'Actual_Calls': actual_calls if pd.notna(actual_calls) else 0,
                        'Forecast_AHT': forecast_aht if pd.notna(forecast_aht) else None,
                        'Actual_AHT': actual_aht if pd.notna(actual_aht) else None,
                        'Actual_Abandon': actual_abandon if pd.notna(actual_abandon) else 0
                    })
            except IndexError:
                continue
    
    return pd.DataFrame(records)

In [ ]:
def process_excel_file(file_path):
    """
    Process a single Excel file with multiple LOB sheets.
    Each file contains 2 weeks of data.
    """
    print(f"Processing: {Path(file_path).name}")
    xl = pd.ExcelFile(file_path)
    
    all_data = []
    
    for sheet_name in xl.sheet_names:
        print(f"  Processing LOB: {sheet_name}")
        
        # Read the sheet
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
        
        # Find week start dates (they appear in column 0)
        week_starts = []
        week_start_rows = []
        
        for idx, val in df.iloc[:, 0].items():
            if pd.notna(val) and isinstance(val, (datetime, pd.Timestamp)):
                week_starts.append(pd.to_datetime(val))
                week_start_rows.append(idx)
        
        # Process each week
        for i, (week_start, start_row) in enumerate(zip(week_starts, week_start_rows)):
            # Determine end row (next week start or end of data)
            if i + 1 < len(week_start_rows):
                end_row = week_start_rows[i + 1]
            else:
                end_row = len(df)
            
            # Extract this week's data
            week_df = df.iloc[start_row:end_row].copy()
            week_df = week_df.reset_index(drop=True)
            
            # Parse the week
            parsed_df = parse_interval_sheet(week_df, sheet_name, week_start)
            
            if len(parsed_df) > 0:
                all_data.append(parsed_df)
    
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()

In [ ]:
# Process all Excel files in the directory
all_interval_data = []

for file_name in os.listdir(data_dir):
    if file_name.endswith('.xlsx') and not file_name.startswith('~'):
        file_path = os.path.join(data_dir, file_name)
        df = process_excel_file(file_path)
        if len(df) > 0:
            all_interval_data.append(df)

# Combine all data
if all_interval_data:
    interval_df = pd.concat(all_interval_data, ignore_index=True)
    print(f"\n=== ETL COMPLETE ===")
    print(f"Total records: {len(interval_df):,}")
    print(f"Date range: {interval_df['Date'].min()} to {interval_df['Date'].max()}")
    print(f"LOBs: {interval_df['LOB'].nunique()}")
else:
    print("No data processed!")

In [ ]:
# Data quality check
print("=== DATA QUALITY ===")
print(f"\nRecords by LOB:")
print(interval_df.groupby('LOB').size().sort_values(ascending=False))

print(f"\nService Level data availability:")
sl_not_null = interval_df['Actual_SL'].notna().sum()
print(f"  Records with SL data: {sl_not_null:,} ({sl_not_null/len(interval_df)*100:.1f}%)")

print(f"\nSample data:")
interval_df.head(10)

In [ ]:
# Filter to records with valid service level data
sl_df = interval_df[interval_df['Actual_SL'].notna()].copy()
print(f"Records with valid SL: {len(sl_df):,}")

# Add SL goal based on LOB
sl_df['SL_Goal'] = sl_df['LOB'].map(sl_goals)

# Calculate deviation from goal (in percentage points)
sl_df['SL_Deviation_Pts'] = (sl_df['Actual_SL'] - sl_df['SL_Goal']) * 100

sl_df.head()

In [ ]:
# Categorize deviations into ranges
def categorize_deviation(row):
    """
    Categorize SL deviation into performance ranges.
    For 90% goal LOBs, >99.5% SL counts as Significantly Above.
    """
    deviation = row['SL_Deviation_Pts']
    actual_sl = row['Actual_SL']
    goal = row['SL_Goal']
    
    # Special handling for 90% goal LOBs - >99.5% is Significantly Above
    if goal == 0.90 and actual_sl > 0.995:
        return 'Significantly Above (+10pt+)'
    
    if deviation >= 10:
        return 'Significantly Above (+10pt+)'
    elif deviation >= 5:
        return 'Slightly Above (+5 to +9pt)'
    elif deviation >= -5:
        return 'Within Range (+/-5pt)'
    elif deviation >= -10:
        return 'Slightly Below (-5 to -9pt)'
    else:
        return 'Significantly Below (-10pt+)'

sl_df['Performance_Category'] = sl_df.apply(categorize_deviation, axis=1)

# Define category order for visualization (bottom to top: red to blue)
category_order = [
    'Significantly Below (-10pt+)',
    'Slightly Below (-5 to -9pt)',
    'Within Range (+/-5pt)',
    'Slightly Above (+5 to +9pt)',
    'Significantly Above (+10pt+)'
]

print("=== OVERALL DISTRIBUTION ===")
print(sl_df['Performance_Category'].value_counts().reindex(category_order))

In [ ]:
# Create summary by LOB
lob_summary = sl_df.groupby(['LOB', 'Performance_Category']).size().unstack(fill_value=0)

# Reorder columns
lob_summary = lob_summary.reindex(columns=category_order, fill_value=0)

# Add totals
lob_summary['Total_Intervals'] = lob_summary.sum(axis=1)

# Add goal column
lob_summary['SL_Goal'] = [f"{sl_goals.get(lob, 0.80)*100:.0f}%" for lob in lob_summary.index]

print("=== INTERVAL COUNTS BY LOB AND CATEGORY ===")
display(lob_summary)

In [ ]:
# Create vertical stacked bar chart (interval counts)
fig, ax = plt.subplots(figsize=(14, 8))

# Get LOBs sorted by total intervals descending
lob_order = lob_summary['Total_Intervals'].sort_values(ascending=False).index.tolist()

# Colors: red → orange → green → light blue → blue (bottom to top)
colors = ['#e74c3c', '#f39c12', '#27ae60', '#5dade2', '#2980b9']

# Prepare data for stacked bar chart (order matches category_order: below at bottom)
plot_data = lob_summary.loc[lob_order, category_order]

# Create vertical stacked bar chart
plot_data.plot(kind='bar', stacked=True, ax=ax, color=colors, edgecolor='white', linewidth=0.5, width=0.8)

# Customize
ax.set_ylabel('Number of 30-Minute Intervals', fontsize=12)
ax.set_xlabel('Line of Business', fontsize=12)
ax.set_title('Service Level Performance by LOB (Jan + Aug 2025 Combined)\n(Intervals categorized by deviation from SL goal)', fontsize=14, fontweight='bold')

# Rotate x-axis labels for readability
plt.xticks(rotation=45, ha='right')

# Add SL goal annotations on top of each bar
for i, lob in enumerate(lob_order):
    goal = sl_goals[lob]
    total = lob_summary.loc[lob, 'Total_Intervals']
    ax.annotate(f'{goal*100:.0f}%', xy=(i, total + 20), ha='center', fontsize=9, color='gray', fontweight='bold')

# Add total count labels
for i, lob in enumerate(lob_order):
    total = lob_summary.loc[lob, 'Total_Intervals']
    ax.annotate(f'n={total}', xy=(i, total + 5), ha='center', fontsize=8, color='black')

# Move legend outside
ax.legend(title='Performance Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

# Add gridlines for readability
ax.yaxis.grid(True, linestyle='--', alpha=0.3)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(f"{output_dir}/sl_performance_by_lob_vertical.png", dpi=150, bbox_inches='tight')
print(f"Saved: {output_dir}/sl_performance_by_lob_vertical.png")
plt.show()

In [ ]:
# Summary table for the report
print("=== COMBINED REPORT: January + August 2025 ===")
print(f"Date Range: {sl_df['Date'].min().strftime('%Y-%m-%d')} to {sl_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Total Intervals Analyzed: {len(sl_df):,}")
print(f"\nLOBs with 80% Goal: {sum(1 for g in sl_goals.values() if g == 0.80)}")
print(f"LOBs with 90% Goal: {sum(1 for g in sl_goals.values() if g == 0.90)}")

# Create a cleaner summary table
summary_table = lob_summary[['Total_Intervals', 'SL_Goal'] + category_order].copy()
summary_table = summary_table.sort_values('Total_Intervals', ascending=False)
print("\n")
display(summary_table)

In [ ]:
# Save ETL'd data and summary
interval_df.to_csv(f"{output_dir}/interval_data_etl.csv", index=False)
lob_summary.to_csv(f"{output_dir}/lob_sl_summary.csv")

print("=== FILES SAVED ===")
print(f"  {output_dir}/interval_data_etl.csv")
print(f"  {output_dir}/lob_sl_summary.csv")
print(f"  {output_dir}/sl_performance_by_lob.png")
print(f"  {output_dir}/sl_performance_pct_by_lob.png")

In [ ]:
# Final summary
print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)

print(f"\nTotal intervals analyzed: {len(sl_df):,}")
print(f"Date range: {sl_df['Date'].min().strftime('%Y-%m-%d')} to {sl_df['Date'].max().strftime('%Y-%m-%d')}")
print(f"\nPerformance Summary (all LOBs):")
for cat in category_order:
    count = (sl_df['Performance_Category'] == cat).sum()
    pct = count / len(sl_df) * 100
    print(f"  {cat}: {count:,} intervals ({pct:.1f}%)")